# 🔬 Continual Anomaly Detection (VisA) - Google Colab

Questo notebook permette di configurare ed eseguire il progetto **Continual Anomaly Detection** su Google Colab, utilizzando il dataset **VisA**.

### Prerequisiti
- Seleziona un runtime con **GPU** (Runtime → Cambia tipo di runtime → T4 GPU)
- Il dataset VisA deve essere caricato su **Google Drive**

## 0. Verifica GPU
Controlliamo che il runtime abbia una GPU disponibile.

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✅ GPU disponibile: {gpu_name}")
else:
    print("⚠️ ATTENZIONE: Nessuna GPU rilevata!")
    print("   Vai su Runtime → Cambia tipo di runtime → T4 GPU")

## 1. Ottenere il Codice del Progetto

Scegli **UNA** delle due opzioni:

### Opzione A: Clonare da GitHub (richiede token se il repo è privato)
### Opzione B: Caricare da Google Drive (se hai il progetto su Drive)

In [ ]:
import os
import subprocess

REPO_DIR = "Continual_Anomaly_Detection"

# ============================================================
# OPZIONE A: Clonare da GitHub
# Se il repo è privato, inserisci il tuo GitHub Personal Access Token
# ============================================================
REPO_URL = "https://github.com/roccovardaro/Continual_Anomaly_Detection.git"
GITHUB_TOKEN = ""  # ⚠️ Inserisci il token qui se il repo è privato

# ============================================================
# OPZIONE B: Caricare da Google Drive
# Se il progetto è già su Drive, imposta il percorso qui
# e commenta l'Opzione A
# ============================================================
# DRIVE_PROJECT_PATH = "/content/drive/MyDrive/Continual_Anomaly_Detection"

# --- Logica di setup (non modificare) ---
if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
    print(f"ℹ️ La directory {REPO_DIR} esiste già.")
    print(f"📂 Working directory: {os.getcwd()}")
elif 'DRIVE_PROJECT_PATH' in dir() and os.path.exists(DRIVE_PROJECT_PATH):
    # Opzione B: copia da Drive a /content/
    import shutil
    print(f"📂 Copio il progetto da Google Drive...")
    shutil.copytree(DRIVE_PROJECT_PATH, REPO_DIR)
    os.chdir(REPO_DIR)
    print(f"✅ Progetto copiato da Drive!")
    print(f"📂 Working directory: {os.getcwd()}")
else:
    # Opzione A: clone da GitHub
    if GITHUB_TOKEN:
        clone_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
    else:
        clone_url = REPO_URL
    
    result = subprocess.run(["git", "clone", clone_url], capture_output=True, text=True)
    
    if result.returncode == 0 and os.path.exists(REPO_DIR):
        os.chdir(REPO_DIR)
        print(f"✅ Repository clonato con successo!")
        print(f"📂 Working directory: {os.getcwd()}")
    else:
        print("❌ ERRORE: Clone fallito!")
        print(f"   {result.stderr.strip()}")
        print()
        print("💡 Soluzioni possibili:")
        print("   1. Se il repo è PRIVATO → imposta GITHUB_TOKEN con un Personal Access Token")
        print("      (GitHub → Settings → Developer settings → Personal access tokens → Tokens (classic))")
        print("   2. Carica il progetto su Google Drive e usa l'OPZIONE B:")
        print("      - De-commenta la riga DRIVE_PROJECT_PATH")
        print("      - Imposta il percorso corretto")
        print("      - Ri-esegui questa cella")

## 2. Installare le Dipendenze
Installiamo le librerie necessarie. PyTorch e torchvision sono già preinstallati su Colab con supporto CUDA, quindi li escludiamo per evitare conflitti.

In [ ]:
!pip install efficientnet_pytorch einops imgaug timm scipy scikit-learn \
    Pillow PyYAML tqdm pandas matplotlib codecarbon umap-learn opencv-python

print("\n✅ Dipendenze installate!")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA disponibile: {torch.cuda.is_available()}")

## 3. Scaricare il Modello Preaddestrato (solo se usi ViT)
Se il config usa `model.name: vit`, il checkpoint ViT-B/16 verrà scaricato nella cartella `checkpoints/`.

**Se usi `convnext` o `resnet`, puoi saltare questa cella.**

In [ ]:
import os

os.makedirs("checkpoints", exist_ok=True)

vit_path = "checkpoints/ViT-B_16.npz"
if not os.path.exists(vit_path):
    print("⬇️ Downloading ViT-B/16 checkpoint...")
    !wget -q --show-progress https://storage.googleapis.com/vit_models/sam/ViT-B_16.npz -P checkpoints/
    print("✅ Checkpoint scaricato!")
else:
    print("ℹ️ Checkpoint ViT-B/16 già presente, skip download.")

## 4. Collegare Google Drive
Montiamo Google Drive per accedere al dataset VisA.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive montato in /content/drive")

## 5. Configurare il Percorso del Dataset VisA

**⚠️ MODIFICA IL PERCORSO QUI SOTTO** con la posizione effettiva del dataset VisA su Google Drive.

Struttura attesa per VisA:
```
VisA/
├── candle/
│   └── Data/
│       ├── Images/
│       │   ├── Normal/   (*.JPG)
│       │   └── Anomaly/  (*.JPG)
│       └── Masks/
│           └── Anomaly/  (*.png)
├── capsules/
├── cashew/
├── chewinggum/
├── fryum/
├── macaroni1/
├── macaroni2/
├── pcb1/
├── pcb2/
├── pcb3/
├── pcb4/
├── pipe_fryum/
└── split_csv/
    └── 1cls.csv
```

**IMPORTANTE**: La cartella `split_csv/` con il file `1cls.csv` deve essere presente nella root del dataset.

In [ ]:
import os

# ============================================================
# ⚠️ MODIFICA QUESTO PERCORSO CON IL TUO PATH SU GOOGLE DRIVE
# ============================================================
DATA_DIR = "/content/drive/MyDrive/VisA"

# Verifica che il percorso esista e contenga le classi VisA attese
VISA_CLASSES = ['candle', 'capsules', 'cashew', 'chewinggum', 'fryum',
                'macaroni1', 'macaroni2', 'pcb1', 'pcb2', 'pcb3', 'pcb4', 'pipe_fryum']

if os.path.exists(DATA_DIR):
    contents = os.listdir(DATA_DIR)
    print(f"✅ DATA_DIR: {DATA_DIR}")
    print(f"   Contenuto ({len(contents)} elementi): {sorted(contents)}")
    
    # Verifica le classi
    missing = [c for c in VISA_CLASSES if c not in contents]
    if missing:
        print(f"\n⚠️ Classi mancanti: {missing}")
    else:
        print(f"\n✅ Tutte le 12 classi VisA trovate!")
    
    # Verifica split_csv
    csv_path = os.path.join(DATA_DIR, 'split_csv', '1cls.csv')
    if os.path.exists(csv_path):
        print(f"✅ File split CSV trovato: {csv_path}")
    else:
        print(f"⚠️ ATTENZIONE: {csv_path} NON TROVATO!")
        print(f"   Il file split_csv/1cls.csv è necessario per il dataset VisA.")
else:
    print(f"⚠️ DATA_DIR: {DATA_DIR} NON TROVATO!")
    print(f"   Assicurati che il percorso sia corretto.")

## 6. Configurazione dell'Esperimento

I parametri dell'esperimento sono definiti nel file `configs/cad.yaml`.

**⚠️ IMPORTANTE**: Assicurati che il file YAML sia configurato per VisA:
- `dataset.name: seq-visa`
- `dataset.n_classes_per_task: 3` (per setting `mul`: 3+3+3+3 = 12 classi)
- `dataset.n_tasks: 4` (per setting `mul`)

**Modifica il file YAML direttamente** prima di eseguire il training.

In [ ]:
# ============================================================
# CONFIGURAZIONE ESPERIMENTO
# ============================================================
CONFIG_FILE = "./configs/cad_VisA.yaml"
DEVICE = "cuda"   # 'cuda' su Colab (default), 'cpu' come fallback
SEED   = 42

# Mostra il contenuto del config YAML
print("📄 Contenuto di", CONFIG_FILE)
print("=" * 50)
with open(CONFIG_FILE, 'r') as f:
    print(f.read())

# Verifica che il config sia impostato per VisA
import yaml
with open(CONFIG_FILE, 'r') as f:
    config = yaml.safe_load(f)

dataset_name = config.get('dataset', {}).get('name', '')
if 'visa' not in dataset_name:
    print("\n⚠️ ATTENZIONE: il config attuale usa il dataset:", dataset_name)
    print("   Per VisA, assicurati di impostare dataset.name a 'seq-visa'")
else:
    print(f"\n✅ Config corretto per VisA (dataset: {dataset_name})")

print("\n📋 Riepilogo:")
print(f"   Config file: {CONFIG_FILE}")
print(f"   Device:      {DEVICE}")
print(f"   Seed:        {SEED}")
print(f"   Data dir:    {DATA_DIR}")

## 7. Eseguire il Training 🚀

Esegui `main.py` con i parametri configurati sopra.

Il training produce:
- **Checkpoint** del modello in `checkpoints/`
- **Statistiche** dei tempi in `training_times.csv`
- **Visualizzazioni** (istogrammi, t-SNE, UMAP) nelle rispettive cartelle

In [ ]:
!python main.py \
    --config-file {CONFIG_FILE} \
    --data_dir "{DATA_DIR}" \
    --device {DEVICE} \
    --seed {SEED}

## 8. Risultati e Visualizzazione
Dopo il training, visualizziamo i risultati prodotti.

In [ ]:
import pandas as pd
import os

# Mostra i tempi di training
csv_path = "training_times.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print("⏱️ Statistiche sui tempi di training:")
    display(df)
    
    # Mostra il tempo totale per task
    task_totals = df[df['Time_Type'] == 'Task_Train_Total']
    if not task_totals.empty:
        print("\n📊 Tempo totale per task:")
        for _, row in task_totals.iterrows():
            mins = row['Duration_Seconds'] / 60
            print(f"   Task {int(row['Task'])}: {row['Duration_Seconds']:.1f}s ({mins:.1f} min)")
else:
    print("⚠️ File training_times.csv non trovato. Esegui prima il training.")

In [ ]:
import glob
from IPython.display import Image, display

# Mostra le visualizzazioni generate (istogrammi, t-SNE, UMAP)
for folder, label in [("hist_results", "📊 Istogrammi"), 
                       ("tsne_results", "🔵 t-SNE"), 
                       ("umap_results", "🟣 UMAP")]:
    images = sorted(glob.glob(f"{folder}/**/*.png", recursive=True))
    if images:
        print(f"\n{label} ({len(images)} immagini):")
        for img_path in images[-6:]:  # mostra le ultime 6
            print(f"  → {img_path}")
            display(Image(filename=img_path, width=600))
    else:
        print(f"\n{label}: nessuna immagine trovata in {folder}/")

## 9. Salvare i Risultati su Google Drive (Opzionale)
Copia checkpoint e risultati sul tuo Google Drive per conservarli.

In [ ]:
import shutil
import os

# ⚠️ Modifica questo percorso con la destinazione su Google Drive
SAVE_DIR = "/content/drive/MyDrive/CAD_results_visa"

os.makedirs(SAVE_DIR, exist_ok=True)

# Copia i risultati
for folder in ["checkpoints", "hist_results", "tsne_results", "umap_results"]:
    if os.path.exists(folder):
        dest = os.path.join(SAVE_DIR, folder)
        if os.path.exists(dest):
            shutil.rmtree(dest)
        shutil.copytree(folder, dest)
        print(f"✅ {folder} → {dest}")

# Copia il CSV dei tempi
if os.path.exists("training_times.csv"):
    shutil.copy2("training_times.csv", SAVE_DIR)
    print(f"✅ training_times.csv → {SAVE_DIR}")

# Copia il config usato
shutil.copy2(CONFIG_FILE, SAVE_DIR)
print(f"✅ {CONFIG_FILE} → {SAVE_DIR}")

print(f"\n🎉 Risultati salvati in {SAVE_DIR}")